# Celsius Holdings (CELH) — Data Pull

Pulls the live financials and market data that feed the valuation model
(`CELH_Valuation_Model.xlsx`).

## Setup

In [1]:
!pip install -q yfinance 2>/dev/null
import warnings; warnings.filterwarnings("ignore")
import yfinance as yf
import pandas as pd

## Pull CELH market data and financials

Pulls price, share count, debt, cash, revenue, and EBIT.

In [2]:
TICKER = "CELH"

BASE_CASE = {  # last-known values (Jun 2026) used if the live pull fails
    "price": 30.50, "shares_mm": 253, "total_debt_mm": 900, "cash_mm": 549,
    "fy_revenue_mm": 2515, "ttm_revenue_mm": 2969, "fy_ebit_mm": 141,
    "adj_ebitda_mm": 500, "adj_eps": 1.34,
}

def pull(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        fin = t.financials
        bs = t.balance_sheet
        out = {}
        out["price"] = info.get("currentPrice") or info.get("regularMarketPrice")
        out["shares_mm"] = (info.get("sharesOutstanding") or 0) / 1e6
        out["total_debt_mm"] = (info.get("totalDebt") or 0) / 1e6
        out["cash_mm"] = (info.get("totalCash") or 0) / 1e6
        out["ttm_revenue_mm"] = (info.get("totalRevenue") or 0) / 1e6
        if fin is not None and "Total Revenue" in fin.index:
            out["fy_revenue_mm"] = float(fin.loc["Total Revenue"].iloc[0]) / 1e6
        if fin is not None and "EBIT" in fin.index:
            out["fy_ebit_mm"] = float(fin.loc["EBIT"].iloc[0]) / 1e6
        if not out.get("price"):
            return None
        return out
    except Exception as e:
        print(f"Live pull failed ({type(e).__name__}).")
        return None

live = pull(TICKER)
if live:
    data = {**BASE_CASE, **{k: v for k, v in live.items() if v}}
    SOURCE = "yfinance (live)"
else:
    data = BASE_CASE
    SOURCE = "base case (Jun 2026)"

print(f"Data source: {SOURCE}\n")
print(f"{'Input':<34}{'Value':>14}")
print("-"*48)
rows = [("Current share price", data["price"], "$"),
        ("Diluted shares (mm)", data["shares_mm"], ""),
        ("Total debt ($mm)", data["total_debt_mm"], ""),
        ("Cash ($mm)", data["cash_mm"], ""),
        ("FY revenue ($mm)", data["fy_revenue_mm"], ""),
        ("TTM revenue ($mm)", data["ttm_revenue_mm"], ""),
        ("FY EBIT ($mm)", data.get("fy_ebit_mm", BASE_CASE["fy_ebit_mm"]), "")]
for label, val, pre in rows:
    print(f"{label:<34}{pre}{val:>13,.2f}")
print(f"\nNet debt ($mm): {data['total_debt_mm']-data['cash_mm']:,.0f}")
print(f"Market cap ($mm): {data['price']*data['shares_mm']:,.0f}")

Data source: yfinance (live)

Input                                      Value
------------------------------------------------
Current share price               $        30.80
Diluted shares (mm)                      255.64
Total debt ($mm)                         675.88
Cash ($mm)                               549.20
FY revenue ($mm)                       2,515.27
TTM revenue ($mm)                      2,968.61
FY EBIT ($mm)                            174.01

Net debt ($mm): 127
Market cap ($mm): 7,874


## Peer multiples for the comps tab

Quick pull of peer valuation multiples.

In [3]:
PEERS = ["MNST", "KDP", "KO", "PEP"]
records = []
for tk in PEERS:
    try:
        info = yf.Ticker(tk).info
        records.append({
            "ticker": tk,
            "EV/EBITDA": info.get("enterpriseToEbitda"),
            "P/E (fwd)": info.get("forwardPE"),
            "EV/Rev": info.get("enterpriseToRevenue"),
        })
    except Exception:
        records.append({"ticker": tk, "EV/EBITDA": None, "P/E (fwd)": None, "EV/Rev": None})

peers_df = pd.DataFrame(records).set_index("ticker")
print("Peer multiples (live; update the Comps sheet's blue cells):")
display(peers_df.round(1))
print("\nMedians:")
print(peers_df.median(numeric_only=True).round(1).to_string())

Peer multiples (live; update the Comps sheet's blue cells):


,EV/EBITDA,P/E (fwd),EV/Rev
ticker,,,
MNST,31.2,35.3,10.0
KDP,18.0,12.2,4.7
KO,22.6,22.8,7.7
PEP,12.9,15.6,2.5



Medians:
EV/EBITDA    20.3
P/E (fwd)    19.2
EV/Rev        6.2
